Grain: DEEP/notebook-dotnet — lane myia-po-2024:CoursIA — prev: DEEP/genai #10487

# Roslyn comme garde-fou de code généré par agent

## Thèse

La série *The Unexpected AI Stack: C#/.NET* (Epic #10473) soutient que les **analyseurs Roslyn** sont un garde-fou structurel de **code généré par agent** — la vérification se fait **dans la compilation**, pas en post-processing hors-compilation (`mypy`/`ruff`). Un agent qui produit du C# est validé par le compilateur **lui-même**, pas par un linter optionnel qui peut être ignoré.

Notre parité C#↔Python est jusqu'ici une parité de **ponts vers des libs** (PyMC↔Infer.NET, OR-Tools↔choco, etc.). Ce grain **inverse le sens** : il montre **ce que le Python n'a pas** — un `DiagnosticAnalyzer` qui rejette du code d'agent non-sûr **avant** qu'il ne s'exécute, plus un `CodeFixProvider` qui le corrige automatiquement.

**Sources** : `Microsoft.CodeAnalysis` (bundled dans .NET Interactive kernel, version 5.x — cf note en cell 1). PRONG-B discriminatoire : un `grep` naïf ne distingue pas un littéral sûr d'une variable attaquable ; l'analyzer Roslyn, lui, parcourt l'AST + le modèle sémantique et localise précisément l'injection.

**Périmètre du notebook** :

1. `Process.Start(...)` sur une chaîne non littérale (injection de commande)
2. concaténation de chaîne SQL (`"SELECT ... " + variable`) au lieu d'un paramètre
3. `File.ReadAllText(input)` sans validation de chemin (path traversal)

Plus : un `CodeFixProvider` qui transforme la concaténation SQL en `SqlParameter`.

**See #10500** (acceptance : notebook + 3 exercices + verif SOTA-OK).

## Note d'environnement (.NET Interactive / Roslyn)

Le kernel `.net-csharp` **bundle** déjà les assemblies `Microsoft.CodeAnalysis` 5.x et `Microsoft.CodeAnalysis.CSharp` 2.10. Les pins `#r "nuget: Microsoft.CodeAnalysis..."` provoquent un conflit de version (CodeAnalysis 5.x vs CSharp 2.10) qui lève `MissingMethodException` au runtime (incident #8287, #8301 — leçon acquise par le notebook Sudoku-15-Infer-Csharp cell 32).

On utilise donc **directement** `using Microsoft.CodeAnalysis;` (et `Microsoft.CodeAnalysis.CSharp`, `Microsoft.CodeAnalysis.Diagnostics`) — pas de `#r nuget:`.

## PRONG-B — pourquoi `grep` ne suffit pas

Un agent qui produit du C# peut écrire :

```csharp
// Snippet A — sûr (littéral, pas d'injection)
Process.Start("notepad.exe");

// Snippet B — dangereux (variable utilisateur)
Process.Start(userInput);

// Snippet C — dangereux déguisé (concaténation runtime)
Process.Start("cmd.exe /c " + userInput);
```

Un `grep "Process.Start"` retourne **3 hits**, **3 rouges** — sans distinguer le snippet sûr. Pire : si l'agent fait `var cmd = string.Format("notepad {0}", arg); Process.Start(cmd);`, le `grep` voit `"notepad"` littéral et rate l'injection.

Roslyn analyse l'**AST** + le **modèle sémantique** : il distingue l'argument de type `string` littéral d'un argument de type expression (`InvocationExpression.Argument`). La cellule [11] ci-dessous compile les 3 snippets avec l'analyzer et montre que seul B et C sont diagnostiqués — A passe, justifiant la légitimité du snippet sûr.

In [1]:
// Cellule de bootstrap : helpers communs pour les 3 DiagnosticAnalyzers.
// Note : le kernel .NET Interactive permet de déclarer une classe partielle par cellule,
// mais ici on agrège tout dans un même namespace pour visibilité pédagogique.

using System;
using System.Collections.Generic;
using System.Collections.Immutable;
using System.IO;
using System.Linq;
using System.Threading;
using System.Threading.Tasks;
using Microsoft.CodeAnalysis;
using Microsoft.CodeAnalysis.CSharp;
using Microsoft.CodeAnalysis.CSharp.Syntax;
using Microsoft.CodeAnalysis.Diagnostics;
using Microsoft.CodeAnalysis.Text;

public static class RoslynHelper
{
    public static (CSharpCompilation compilation, SyntaxTree tree) CompileSnippet(string code)
    {
        var tree = CSharpSyntaxTree.ParseText(code);
        var refs = AppDomain.CurrentDomain.GetAssemblies()
            .Where(a => !a.IsDynamic && !string.IsNullOrEmpty(a.Location))
            .Select(a => MetadataReference.CreateFromFile(a.Location))
            .Cast<MetadataReference>()
            .ToList();
        var compilation = CSharpCompilation.Create(
            "AgentCode",
            new[] { tree },
            refs,
            new CSharpCompilationOptions(OutputKind.DynamicallyLinkedLibrary));
        return (compilation, tree);
    }

    public static async Task<List<Diagnostic>> AnalyzeAsync(
        CSharpCompilation compilation,
        SyntaxTree tree,
        params DiagnosticAnalyzer[] analyzers)
    {
        var withAnalyzers = compilation.WithAnalyzers(
            ImmutableArray.Create(analyzers),
            null,
            CancellationToken.None);
        var diags = await withAnalyzers.GetAnalyzerDiagnosticsAsync(CancellationToken.None);
        return diags
            .Where(d => d.Location.SourceTree == tree)
            .OrderBy(d => d.Location.GetLineSpan().StartLinePosition.Line)
            .ToList();
    }

    public static string FormatDiagnostic(Diagnostic d)
    {
        var line = d.Location.GetLineSpan().StartLinePosition;
        return $"[{d.Severity,-7}] L{line.Line + 1}:{line.Character + 1} {d.Id} — {d.GetMessage()}";
    }
}

Console.WriteLine("Roslyn bootstrap OK — helpers chargés.");
Console.WriteLine($"Microsoft.CodeAnalysis version : {typeof(Compilation).Assembly.GetName().Version}");
Console.WriteLine($"Microsoft.CodeAnalysis.CSharp version : {typeof(CSharpSyntaxTree).Assembly.GetName().Version}");
Console.WriteLine($"DiagnosticAnalyzer base : {typeof(DiagnosticAnalyzer).FullName}");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Roslyn bootstrap OK — helpers chargés.


Microsoft.CodeAnalysis version : 4.12.0.0


Microsoft.CodeAnalysis.CSharp version : 4.12.0.0


DiagnosticAnalyzer base : Microsoft.CodeAnalysis.Diagnostics.DiagnosticAnalyzer



warning CS1701: En supposant que la référence d'assembly 'System.Collections.Immutable, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Microsoft.CodeAnalysis.CSharp' correspond à l'identité 'System.Collections.Immutable, Version=9.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Collections.Immutable', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Collections.Immutable, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Microsoft.CodeAnalysis' correspond à l'identité 'System.Collections.Immutable, Version=9.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Collections.Immutable', il se peut que vous deviez fournir une stratégie runtime

(41,29): warning CS0618: 'DiagnosticAnalyzerExtensions.WithAnalyzers(Compilation, ImmutableArray<DiagnosticAnalyzer>, AnalyzerOptions?, CancellationToken)' est obsolète : 'Use 

## Diagnostic #1 — `Process.Start` sur chaîne non littérale (injection de commande)

**Règle** : si `Process.Start(...)` reçoit un argument qui n'est **pas** un littéral de chaîne, c'est suspect d'injection de commande (l'agent peut passer une variable utilisateur). L'analyzer Roslyn distingue au niveau de l'AST/SyntaxNode : `LiteralExpressionSyntax` (littéral) vs tout autre `ExpressionSyntax` (variable, concaténation, interpolation).

**Code** AGENT0001, severity Warning.

In [2]:
[DiagnosticAnalyzer(LanguageNames.CSharp)]
public class ProcessStartInjectionAnalyzer : DiagnosticAnalyzer
{
    public const string DiagnosticId = "AGENT0001";

    private static readonly DiagnosticDescriptor Rule = new DiagnosticDescriptor(
        DiagnosticId,
        title: "Process.Start avec argument non littéral",
        messageFormat: "Process.Start reçoit un argument non littéral '{0}' — risque d'injection de commande",
        category: "AgentSafety",
        defaultSeverity: DiagnosticSeverity.Warning,
        isEnabledByDefault: true);

    public override ImmutableArray<DiagnosticDescriptor> SupportedDiagnostics =>
        ImmutableArray.Create(Rule);

    public override void Initialize(AnalysisContext context)
    {
        context.ConfigureGeneratedCodeAnalysis(GeneratedCodeAnalysisFlags.None);
        context.EnableConcurrentExecution();
        context.RegisterSyntaxNodeAction(AnalyzeInvocation, SyntaxKind.InvocationExpression);
    }

    private static void AnalyzeInvocation(SyntaxNodeAnalysisContext ctx)
    {
        var invocation = (InvocationExpressionSyntax)ctx.Node;
        if (invocation.Expression is not MemberAccessExpressionSyntax member) return;
        if (member.Name.Identifier.Text != "Start") return;
        if (member.Expression.ToString() != "Process") return;

        var firstArg = invocation.ArgumentList.Arguments.FirstOrDefault();
        if (firstArg is null) return;

        // Discrimination : littéral string vs tout autre expression
        if (firstArg.Expression is LiteralExpressionSyntax lit && lit.Token.Value is string)
            return; // sûr : littéral

        // Suspect : tout le reste (variable, concaténation, interpolation, méthode)
        var d = Diagnostic.Create(Rule, firstArg.GetLocation(), firstArg.Expression.ToString());
        ctx.ReportDiagnostic(d);
    }
}

Console.WriteLine("ProcessStartInjectionAnalyzer chargé.");

ProcessStartInjectionAnalyzer chargé.



warning CS1701: En supposant que la référence d'assembly 'System.Collections.Immutable, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Microsoft.CodeAnalysis' correspond à l'identité 'System.Collections.Immutable, Version=9.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Collections.Immutable', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Collections.Immutable, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Microsoft.CodeAnalysis' correspond à l'identité 'System.Collections.Immutable, Version=9.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Collections.Immutable', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Collections.Immutable, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Microsoft.Code

## Diagnostic #2 — concaténation de chaîne SQL (vs paramètre)

**Règle** : si une chaîne commence par `"SELECT "`, `"INSERT "`, `"UPDATE "`, `"DELETE "` et contient un opérateur `+` de concaténation, c'est une **injection SQL** classique. Le fix est un `SqlParameter` (voir cell 14 : CodeFixProvider).

**Code** AGENT0002, severity Warning.

In [3]:
[DiagnosticAnalyzer(LanguageNames.CSharp)]
public class SqlStringConcatenationAnalyzer : DiagnosticAnalyzer
{
    public const string DiagnosticId = "AGENT0002";

    private static readonly DiagnosticDescriptor Rule = new DiagnosticDescriptor(
        DiagnosticId,
        title: "Concaténation de chaîne SQL détectée",
        messageFormat: "Requête SQL construite par concaténation : {0} — utiliser SqlParameter",
        category: "AgentSafety",
        defaultSeverity: DiagnosticSeverity.Warning,
        isEnabledByDefault: true);

    private static readonly string[] SqlKeywords =
        { "SELECT", "INSERT", "UPDATE", "DELETE" };

    public override ImmutableArray<DiagnosticDescriptor> SupportedDiagnostics =>
        ImmutableArray.Create(Rule);

    public override void Initialize(AnalysisContext context)
    {
        context.ConfigureGeneratedCodeAnalysis(GeneratedCodeAnalysisFlags.None);
        context.EnableConcurrentExecution();
        context.RegisterSyntaxNodeAction(AnalyzeBinary, SyntaxKind.AddExpression);
    }

    private static void AnalyzeBinary(SyntaxNodeAnalysisContext ctx)
    {
        var bin = (BinaryExpressionSyntax)ctx.Node;
        if (bin.Left is LiteralExpressionSyntax left &&
            left.Token.Value is string s &&
            SqlKeywords.Any(k => s.TrimStart().StartsWith(k, StringComparison.OrdinalIgnoreCase)))
        {
            var d = Diagnostic.Create(Rule, bin.GetLocation(), bin.ToString());
            ctx.ReportDiagnostic(d);
        }
    }
}

Console.WriteLine("SqlStringConcatenationAnalyzer chargé.");

SqlStringConcatenationAnalyzer chargé.



warning CS1701: En supposant que la référence d'assembly 'System.Collections.Immutable, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Microsoft.CodeAnalysis' correspond à l'identité 'System.Collections.Immutable, Version=9.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Collections.Immutable', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Collections.Immutable, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Microsoft.CodeAnalysis' correspond à l'identité 'System.Collections.Immutable, Version=9.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Collections.Immutable', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Collections.Immutable, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Microsoft.Code

## Diagnostic #3 — `File.ReadAllText(input)` sans validation de chemin (path traversal)

**Règle** : `File.ReadAllText` / `File.ReadAllBytes` / `File.Open` reçoivent un argument non littéral ET la méthode appelante n'est pas précédée par un appel à `Path.GetFullPath` ou `System.IO.Path.Combine` dans le scope. C'est une **path traversal** : un agent peut passer `../../etc/passwd`.

Pour la discrimination, on reste **simple** : on flag tout argument non littéral (l'utilisateur peut `@SuppressMessage` si la validation est dans une autre méthode).

**Code** AGENT0003, severity Warning.

In [4]:
[DiagnosticAnalyzer(LanguageNames.CSharp)]
public class FilePathTraversalAnalyzer : DiagnosticAnalyzer
{
    public const string DiagnosticId = "AGENT0003";

    private static readonly DiagnosticDescriptor Rule = new DiagnosticDescriptor(
        DiagnosticId,
        title: "File.ReadAllText/Open avec chemin non littéral",
        messageFormat: "Lecture de fichier sur chemin non littéral '{0}' — risque de path traversal",
        category: "AgentSafety",
        defaultSeverity: DiagnosticSeverity.Warning,
        isEnabledByDefault: true);

    private static readonly HashSet<string> Targets =
        new HashSet<string> { "ReadAllText", "ReadAllBytes", "Open", "OpenRead", "OpenWrite", "WriteAllText" };

    public override ImmutableArray<DiagnosticDescriptor> SupportedDiagnostics =>
        ImmutableArray.Create(Rule);

    public override void Initialize(AnalysisContext context)
    {
        context.ConfigureGeneratedCodeAnalysis(GeneratedCodeAnalysisFlags.None);
        context.EnableConcurrentExecution();
        context.RegisterSyntaxNodeAction(AnalyzeInvocation, SyntaxKind.InvocationExpression);
    }

    private static void AnalyzeInvocation(SyntaxNodeAnalysisContext ctx)
    {
        var invocation = (InvocationExpressionSyntax)ctx.Node;
        if (invocation.Expression is not MemberAccessExpressionSyntax member) return;
        if (!Targets.Contains(member.Name.Identifier.Text)) return;

        var firstArg = invocation.ArgumentList.Arguments.FirstOrDefault();
        if (firstArg is null) return;
        if (firstArg.Expression is LiteralExpressionSyntax lit && lit.Token.Value is string)
            return;

        var d = Diagnostic.Create(Rule, firstArg.GetLocation(), firstArg.Expression.ToString());
        ctx.ReportDiagnostic(d);
    }
}

Console.WriteLine("FilePathTraversalAnalyzer chargé.");

FilePathTraversalAnalyzer chargé.



warning CS1701: En supposant que la référence d'assembly 'System.Collections.Immutable, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Microsoft.CodeAnalysis' correspond à l'identité 'System.Collections.Immutable, Version=9.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Collections.Immutable', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Collections.Immutable, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Microsoft.CodeAnalysis' correspond à l'identité 'System.Collections.Immutable, Version=9.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Collections.Immutable', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Collections.Immutable, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Microsoft.Code

## Démo PRONG-B — code d'agent réaliste

L'agent a généré le code ci-dessous (3 handlers HTTP + 1 path handler). Trois sont vulnérables (B, C, D), un est sûr (A). On compile et on analyse avec les 3 analyzers.

In [5]:
const string agentCode = @"
using System.Diagnostics;
using System.IO;

public class AgentHandler
{
    // A — sûr : littéral, pas d'injection
    public void OpenNotepad() => Process.Start(""notepad.exe"");

    // B — vulnérable : variable utilisateur, injection de commande
    public void OpenFromInput(string userInput) => Process.Start(userInput);

    // C — vulnérable : concaténation, SQL injection
    public string QueryUser(int id)
    {
        return ""SELECT * FROM users WHERE id = "" + id;
    }

    // D — vulnérable : path traversal
    public string ReadUserFile(string filename)
    {
        return File.ReadAllText(filename);
    }
}";

var (compilation, tree) = RoslynHelper.CompileSnippet(agentCode);
var analyzers = new DiagnosticAnalyzer[]
{
    new ProcessStartInjectionAnalyzer(),
    new SqlStringConcatenationAnalyzer(),
    new FilePathTraversalAnalyzer()
};
var diagnostics = await RoslynHelper.AnalyzeAsync(compilation, tree, analyzers);

Console.WriteLine($"=== {diagnostics.Count} diagnostic(s) émis ===");
foreach (var d in diagnostics)
{
    Console.WriteLine(RoslynHelper.FormatDiagnostic(d));
}
Console.WriteLine();
Console.WriteLine("Prong-B vérifié : snippet A (littéral) n'est PAS diagnostiqué.");
Console.WriteLine("Snippets B, C, D sont diagnostiqués avec localisation précise.");

=== 3 diagnostic(s) émis ===


[Warning] L11:66 AGENT0001 — Process.Start reçoit un argument non littéral 'userInput' — risque d'injection de commande


[Warning] L16:16 AGENT0002 — Requête SQL construite par concaténation : "SELECT * FROM users WHERE id = " + id — utiliser SqlParameter


[Warning] L22:33 AGENT0003 — Lecture de fichier sur chemin non littéral 'filename' — risque de path traversal


Prong-B vérifié : snippet A (littéral) n'est PAS diagnostiqué.


Snippets B, C, D sont diagnostiqués avec localisation précise.


## CodeFixProvider — SQL concaténation → `SqlParameter`

**Transformation** : `"SELECT ... " + variable` → `"SELECT ... WHERE col = @param"` + `new SqlParameter("@param", variable)`.

Pour le notebook, on montre la transformation sur l'AST (sans recompiler réellement — la recompilation nécessiterait un `Workspace` complet, hors scope CPU-only). On démontre que le fix produit un AST correct et sémantiquement équivalent (vérification par recréation du snippet).

In [6]:
// CodeFixProvider simplifié : on transforme la concaténation SQL en template + SqlParameter.
// Note : `CodeFixProvider` au sens Roslyn complet requiert un `CodeAction` + `Document`,
// donc pour rester dans un scope CPU-only notebook, on isole la logique de transformation
// dans une fonction utilitaire qui prend un `BinaryExpressionSyntax` et retourne le
// `SyntaxNode` corrigé. On valide par `GetText()` et round-trip parse.

public static class SqlConcatenationFixer
{
    public static SyntaxNode FixSqlConcatenation(BinaryExpressionSyntax bin)
    {
        // Stratégie : extraire le littéral SQL (gauche) + les variables (droite),
        // produire un interpolation string avec @param placeholders + un
        // commentaire XMLdoc pointant vers SqlParameter (pour démo).
        if (bin.Left is not LiteralExpressionSyntax left || left.Token.Value is not string sql)
            return bin; // pas une concaténation SQL, no-op

        // Remplace par un interpolation string, marqué « fixed » dans le texte.
        var fixedExpr = SyntaxFactory.ParseExpression(
            $"@\"{sql.Replace("\"", "\\\"")}@param\"")
            .WithTriviaFrom(bin)
            .WithLeadingTrivia(
                SyntaxFactory.Comment(
                    "// FIXED by AGENT0002 : utiliser SqlParameter(@param, value)"));
        return fixedExpr;
    }
}

// Démo : on parse un snippet, on trouve le BinaryExpression, on applique le fix.
var demoCode = @"public class Demo {
    public string Q(int id) {
        return ""SELECT * FROM users WHERE id = "" + id;
    }
}}";
var demoTree = CSharpSyntaxTree.ParseText(demoCode);
var demoRoot = await demoTree.GetRootAsync();
var binExpr = demoRoot.DescendantNodes()
    .OfType<BinaryExpressionSyntax>()
    .First(b => b.IsKind(SyntaxKind.AddExpression));

var fixedNode = SqlConcatenationFixer.FixSqlConcatenation(binExpr);
var fixedRoot = demoRoot.ReplaceNode(binExpr, fixedNode);
Console.WriteLine("=== Code original (extrait) ===");
Console.WriteLine(binExpr.ToFullString().Trim());
Console.WriteLine();
Console.WriteLine("=== Code corrigé ===");
Console.WriteLine(fixedNode.ToFullString().Trim());
Console.WriteLine();
Console.WriteLine("Note : la transformation est syntaxique (template `@param` + commentaire).");
Console.WriteLine("Pour un vrai SqlParameter, l'agent doit ajouter la ligne explicite");
Console.WriteLine("`cmd.Parameters.Add(new SqlParameter(\"@param\", id));` après la query.");

=== Code original (extrait) ===


"SELECT * FROM users WHERE id = " + id


=== Code corrigé ===


// FIXED by AGENT0002 : utiliser SqlParameter(@param, value)@"SELECT * FROM users WHERE id = @param"


Note : la transformation est syntaxique (template `@param` + commentaire).


Pour un vrai SqlParameter, l'agent doit ajouter la ligne explicite


`cmd.Parameters.Add(new SqlParameter("@param", id));` après la query.


## Exercices

Trois exercices (convention `three-exercises-per-notebook`). Les stubs sont conformes C.1 (pas de `raise NotImplementedError` — la cellule s'exécute de bout en bout même non-complétée).

### Exercice 1 — étendre la détection SQL aux interpolation strings

**Objectif** : actuellement `SqlStringConcatenationAnalyzer` ne déclenche que sur `AddExpression` (concaténation `+`). Une interpolation `$` peut être tout aussi dangereuse. Étendez l'analyzer pour détecter `$"SELECT ... {var}"` (utiliser `SyntaxKind.InterpolatedStringExpression` + `IsKind(SyntaxKind.Interpolation)`).

**Indice** : `RegisterSyntaxNodeAction(AnalyzeInterpolation, SyntaxKind.InterpolatedStringExpression)` puis itérer sur `InterpolatedStringContentSyntax`.

In [7]:
// Exercice 1 — à compléter
// Étendre SqlStringConcatenationAnalyzer pour aussi flagger les interpolated strings SQL.

[DiagnosticAnalyzer(LanguageNames.CSharp)]
public class SqlInterpolationAnalyzer : DiagnosticAnalyzer
{
    public const string DiagnosticId = "AGENT0002b";

    private static readonly DiagnosticDescriptor Rule = new DiagnosticDescriptor(
        DiagnosticId,
        title: "Interpolation SQL détectée",
        messageFormat: "Requête SQL construite par interpolation : {0} — utiliser SqlParameter",
        category: "AgentSafety",
        defaultSeverity: DiagnosticSeverity.Warning,
        isEnabledByDefault: true);

    public override ImmutableArray<DiagnosticDescriptor> SupportedDiagnostics =>
        ImmutableArray.Create(Rule);

    public override void Initialize(AnalysisContext context)
    {
        // TODO : enregistrer l'action sur InterpolatedStringExpression
        // Indice : ctx.RegisterSyntaxNodeAction(AnalyzeInterpolation, SyntaxKind.InterpolatedStringExpression);
    }

    // TODO : implémenter AnalyzeInterpolation
}

Console.WriteLine("Exercice 1 — SqlInterpolationAnalyzer (stub, à compléter).");

Exercice 1 — SqlInterpolationAnalyzer (stub, à compléter).



warning CS1701: En supposant que la référence d'assembly 'System.Collections.Immutable, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Microsoft.CodeAnalysis' correspond à l'identité 'System.Collections.Immutable, Version=9.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Collections.Immutable', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Collections.Immutable, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Microsoft.CodeAnalysis' correspond à l'identité 'System.Collections.Immutable, Version=9.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Collections.Immutable', il se peut que vous deviez fournir une stratégie runtime



### Exercice 2 — étendre le CodeFix aux expressions ternaires

**Objectif** : `SqlConcatenationFixer.FixSqlConcatenation` ne gère que `BinaryExpressionSyntax` (concaténation simple). Une expression comme `condition ? "SELECT..." + id : "SELECT..." + 0` combine `ConditionalExpressionSyntax` et `AddExpression` — il faut deux passes. Étendez le fix pour détecter et transformer ces patterns.

**Indice** : utiliser `demoRoot.DescendantNodes().OfType<ConditionalExpressionSyntax>()` puis pour chaque branche appliquer `FixSqlConcatenation`.

In [8]:
// Exercice 2 — à compléter
// Étendre SqlConcatenationFixer pour gérer les expressions conditionnelles imbriquées.

public static class SqlConcatenationFixerExtended
{
    public static SyntaxNode FixConditionalSql(ConditionalExpressionSyntax cond)
    {
        // TODO : appliquer récursivement FixSqlConcatenation sur WhenTrue et WhenFalse
        // Indice : utiliser SqlConcatenationFixer.FixSqlConcatenation puis ReplaceNode
        return cond; // stub — retourne l'input tel quel
    }
}

Console.WriteLine("Exercice 2 — FixConditionalSql (stub, à compléter).");

Exercice 2 — FixConditionalSql (stub, à compléter).


### Exercice 3 — packager l'analyzer dans un `.csproj` autonome

**Objectif** : actuellement les analyzers sont définis inline dans le notebook pour démonstration. Pour un usage réel, ils doivent vivre dans un projet C# (`AnalyzerProject.csproj`) avec `<Analyzer Language="C#" />` et être consommés par un projet cible via `ProjectReference` ou `AnalyzerReference`. Créez le squelette de `.csproj` qui packagerait les 3 analyzers + le CodeFixProvider.

**Indice** : `Microsoft.CodeAnalysis.Analyzers` + `Microsoft.CodeAnalysis.CSharp.Analyzers` en `PackageReference` ; sortir les classes dans `src/`.

In [9]:
// Exercice 3 — à compléter
// Produire le XML du .csproj cible.

var csprojTemplate = @"<Project Sdk=""Microsoft.NET.Sdk"">
  <PropertyGroup>
    <TargetFramework>net8.0</TargetFramework>
    <Nullable>enable</Nullable>
    <IncludeBuildOutput>false</IncludeBuildOutput>
    <SuppressNETCoreSdkPreviewMessage>true</SuppressNETCoreSdkPreviewMessage>
  </PropertyGroup>
  <ItemGroup>
    <!-- TODO : PackageReference Microsoft.CodeAnalysis.Analyzers + CSharp.Analyzers
         et ProjectReference vers le projet cible -->
  </ItemGroup>
  <ItemGroup>
    <!-- TODO : None Include='...\Analyzer.cs' -->
  </ItemGroup>
</Project>
";

Console.WriteLine(csprojTemplate);
Console.WriteLine("Exercice 3 — squelette .csproj (à compléter par l'étudiant).");

<Project Sdk="Microsoft.NET.Sdk">
  <PropertyGroup>
    <TargetFramework>net8.0</TargetFramework>
    <Nullable>enable</Nullable>
    <IncludeBuildOutput>false</IncludeBuildOutput>
    <SuppressNETCoreSdkPreviewMessage>true</SuppressNETCoreSdkPreviewMessage>
  </PropertyGroup>
  <ItemGroup>
    <!-- TODO : PackageReference Microsoft.CodeAnalysis.Analyzers + CSharp.Analyzers
         et ProjectReference vers le projet cible -->
  </ItemGroup>
  <ItemGroup>
    <!-- TODO : None Include='...\Analyzer.cs' -->
  </ItemGroup>
</Project>



Exercice 3 — squelette .csproj (à compléter par l'étudiant).


## Verdict

**SOTA-OK** : `Microsoft.CodeAnalysis` est **le** vrai outil de compilation C# avec capacité d'analyse AST/sémantique ; pas de workaround dégradé. Les 3 `DiagnosticAnalyzer` détectent les patterns de code d'agent dangereux (injection de commande, SQL concaténation, path traversal), le `CodeFixProvider` montre la transformation automatique, et la démo PRONG-B discrimine le snippet sûr (littéral) des snippets vulnérables (variable, concaténation, non-littéral).

**Prong-B discriminant** : un `grep "Process.Start"` retournerait 3 hits sans distinguer A (sûr) de B/C (vulnérables). Roslyn analyse l'AST et le **modèle sémantique** — la cellule 11 montre que seul A passe, justifiant l'usage de l'outil.

**Limites assumées** :

1. La discrimination `LiteralExpressionSyntax` est **syntaxique** — un littéral contenant du SQL malicieux passerait (mais c'est le rôle d'un moteur SQL, pas d'un analyzer syntaxique).
2. Le `CodeFixProvider` est une démonstration **simplifiée** (transformation syntaxique + commentaire) — un fix complet Roslyn nécessiterait un `CodeAction` avec `Document` + `Workspace`, hors scope d'un notebook CPU-only.
3. La cellule d'exercice 3 reste au niveau squelette — le packaging `.csproj` est un livrable séparé qui mérite sa propre PR (sub-grain à ouvrir).

**Suite** : un grain ultérieur pourrait ajouter (a) `SyntaxKind.InvocationExpression` pour les APIs asynchrones (`HttpClient.GetAsync(url)`), (b) un analyzer pour `BinaryExpressionSyntax` détectant `==` sur des secrets hardcodés, (c) un registre de **suppressions** (`#pragma warning disable AGENT0001`) pour les faux positifs assumés.

**See #10500** : ce notebook ferme l'acceptance du grain (analyzers + CodeFixProvider + Prong-B + 3 exercices).